# 🌐 Chạy Trọn Bộ Dự Án Multi-Agent RAG (6 Modes + Web UI Demo2 + GPU Worker)
**Tự động Clone GitHub + Tải Trọng Số Google Drive + Xuất Link Web Cloudflare Tunnel**

Notebook này cho phép chạy **toàn bộ hệ thống** (gồm cả GPU Worker lẫn Web Workspace `http://localhost:8000/demo2/`) trực tiếp trên Kaggle hoặc Google Colab mà không cần cài đặt thủ công.


In [ ]:
# BƯỚC 1: TỰ ĐỘNG CLONE DỰ ÁN TỪ GITHUB & CÀI ĐẶT THƯ VIỆN
import os
import subprocess

REPO_URL = "https://github.com/ducmanh-jr/Multi-Agent_RAG.git"
PROJECT_DIR = "/content/Multi-Agent_RAG" if os.path.exists("/content") else "/kaggle/working/Multi-Agent_RAG"

if not os.path.exists(PROJECT_DIR):
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
else:
    subprocess.run(["git", "-C", PROJECT_DIR, "pull"], check=False)

os.chdir(PROJECT_DIR)
!pip install -q -r requirements.txt
!pip install -q fastapi uvicorn pyngrok gdown transformers peft bitsandbytes accelerate sentence-transformers rank_bm25 pymupdf
print("[OK] Đã cài đặt xong môi trường tại:", os.getcwd())


In [ ]:
# BƯỚC 2: TỰ ĐỘNG NẠP TRỌNG SỐ LORA TỪ GOOGLE DRIVE HOẶC TRONG REPO
import gdown

DRIVE_FILE_ID = ""  # (Tùy chọn) Điền File ID của weights_bundle.zip trên Google Drive nếu muốn tải bản mới nhất
WEIGHTS_DIR = os.path.join(PROJECT_DIR, "training_lab", "weights_drive_bundle")
os.makedirs(WEIGHTS_DIR, exist_ok=True)

if DRIVE_FILE_ID:
    zip_out = os.path.join(WEIGHTS_DIR, "weights_bundle.zip")
    gdown.download(id=DRIVE_FILE_ID, output=zip_out, quiet=False)
    !unzip -q -o {zip_out} -d {WEIGHTS_DIR}
    print("[OK] Đã tải và giải nén weights_bundle.zip từ Google Drive!")
else:
    print("[OK] Sử dụng trọng số có sẵn trong training_lab/weights_drive_bundle/")


In [ ]:
# BƯỚC 3: KHỞI ĐỘNG GPU WORKER (PORT 8001) + WEB SERVER CHÍNH (PORT 8000) + CLOUDFLARE TUNNEL
import time
import re
import subprocess

# Cài đặt cloudflared
if not os.path.exists("/usr/local/bin/cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

# 1. Khởi chạy Kaggle GPU Worker ở cổng 8001 (nếu có GPU)
os.environ["WEIGHTS_DIR"] = WEIGHTS_DIR
worker_proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "kaggle.kaggle_server:app", "--host", "0.0.0.0", "--port", "8001"],
    cwd=PROJECT_DIR,
)
os.environ["REMOTE_INFERENCE_URL"] = "http://127.0.0.1:8001"

# 2. Khởi chạy FastAPI Web Workspace chính ở cổng 8000
web_proc = subprocess.Popen(
    ["python", "main.py", "--serve"],
    cwd=os.path.join(PROJECT_DIR, "main"),
)
time.sleep(5)

# 3. Mở Cloudflare Tunnel ra Internet cho cổng 8000 (Giao diện Web Demo2)
tunnel_proc = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
for _ in range(60):
    line = tunnel_proc.stdout.readline()
    m = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)
    if m:
        public_url = m.group(0)
        break

print("=" * 75)
print(f"🎉 DỰ ÁN ĐÃ CHẠY THÀNH CÔNG TRÊN ĐÁM MÂY!")
print(f"👉 Truy cập Giao diện Web Demo2 tại: {public_url}/demo2/")
print(f"👉 API Docs: {public_url}/docs")
print("=" * 75)

try:
    web_proc.wait()
except KeyboardInterrupt:
    web_proc.terminate()
    worker_proc.terminate()
    tunnel_proc.terminate()
